In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader
from tqdm import tqdm
import math

In [24]:
#130 million parmaeters
class Config:
    vocab_size = 50257
    context_length = 512
    hidden_size = 768
    dim = 768
    emb_dim = 768
    layers = 18
    heads = 16
    num_heads = 16
    steps = 150
    batch_size = 512
    lr = 1e-4
    weight_decay = 0.01
    device = "cuda" if torch.cuda.is_available() else "cpu"

In [25]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
tokenizer.pad_token = tokenizer.eos_token

In [26]:
def get_dataloader():
    dataset = load_dataset("openwebtext", split="train[:30%]") #Trains on 30% of data

    def tokenize(example):
        return tokenizer(
            example["text"],
            truncation=True,
            padding="max_length",
            max_length=Config.context_length,

        )

    dataset = dataset.map(tokenize, batched=True)
    dataset.set_format(type="torch", columns=["input_ids"])

    loader = DataLoader(
        dataset,
        batch_size=Config.batch_size,
        shuffle=True
    )
    return loader

In [27]:
class Diffusion:
    def __init__(self, steps):
        self.steps = steps
        # Cosine schedule is superior for text
        self.beta = self.cosine_beta_schedule(steps).to(Config.device)
        self.alpha = 1.0 - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)

    #This is the cosine scheduler
    def cosine_beta_schedule(self, steps, s=0.008):
        indices = torch.arange(steps + 1)
        alphas_cumprod = torch.cos(((indices / steps) + s) / (1 + s) * torch.pi * 0.5) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return torch.clip(betas, 0.0001, 0.9999)


    #This adds noise to matrix
    def add_noise(self, embedded_tokens, timestamp):
        noise = torch.randn_like(embedded_tokens) #create a matrix of random noise in the same shape of the target embedding
        sqrt_alpha_hat = torch.sqrt(self.alpha_hat[timestamp]).view(-1, 1, 1) #create noise and signal
        sqrt_one_minus_alpha_hat = torch.sqrt(1- self.alpha_hat[timestamp]).view(-1, 1, 1)
        return sqrt_alpha_hat * embedded_tokens+ sqrt_one_minus_alpha_hat * noise, noise #returns the corrupted matrix and the noise

    def denoise_step(self, x_t, pred_x0, t):
        """Standard DDPM sampling step to get x_{t-1}"""
        # Get alpha and alpha_hat for the current timestamp
        alpha = self.alpha[t].view(-1, 1, 1)
        alpha_hat = self.alpha_hat[t].view(-1, 1, 1)
        beta = self.beta[t].view(-1, 1, 1)

        # Calculate the mean for x_{t-1}
        # This formula derives the posterior mean using pred_x0
        if t > 0:
            alpha_hat_prev = self.alpha_hat[t-1].view(-1, 1, 1)
            # Posterior mean coefficients
            coeff1 = (torch.sqrt(alpha_hat_prev) * beta) / (1 - alpha_hat)
            coeff2 = (torch.sqrt(alpha) * (1 - alpha_hat_prev)) / (1 - alpha_hat)
            mean = coeff1 * pred_x0 + coeff2 * x_t

            # Add Langevin dynamics noise (standard DDPM variance)
            noise = torch.randn_like(x_t)
            sigma = torch.sqrt(beta) # Simple variance proxy
            return mean + sigma * noise
        else:
            return pred_x0 # Final step is just the clean prediction

In [28]:
import torch.nn as nn
from transformers import AutoModel


#use pre trained transformer for embedding as it already understands grammer
class DiffusionTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()

        # Frozen RoBERTa embedding layer
        self.encoder = AutoModel.from_pretrained("roberta-base")
        self.embedding = self.encoder.get_input_embeddings()

        for param in self.embedding.parameters():
            param.requires_grad = False

        # Positional embeddings
        self.pos_emb = nn.Parameter(
            torch.randn(1, config.context_length, config.hidden_size)
        )

        # Input projection
        self.input_projection = nn.Linear(config.hidden_size, config.hidden_size)

        # Improved time embedding
        self.time_mlp = nn.Sequential(
        nn.Linear(1, config.hidden_size),
        nn.GELU(),
        nn.Linear(config.hidden_size, config.hidden_size)
        )

        # Transformer blocks
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=config.hidden_size,
                nhead=config.num_heads,
                batch_first=True
            )
            for _ in range(config.layers)
        ])

        # Output projection
        self.final_layer = nn.Linear(config.hidden_size, config.hidden_size)

    def embed(self, tokens):
        return self.embedding(tokens)

    def forward(self, x, t):

        # Project embeddings
        x = self.input_projection(x)

        # Add positional embeddings
        x = x + self.pos_emb[:, :x.size(1)]

        # Normalize timestep
        t = t / Config.steps

        # Time embedding
        t_emb = self.time_mlp(t.float().unsqueeze(-1)).unsqueeze(1)

        # Add timestep conditioning
        x = x + t_emb

        # Transformer layers
        for block in self.blocks:
            x = block(x)

        return self.final_layer(x)

In [32]:
def embedding_to_tokens(embeddings, model):
  vocab_embeddings = model.embedding.weight  # RoBERTa vocab embeddings
  similarity = torch.matmul(embeddings, vocab_embeddings.T)
  tokens = similarity.argmax(dim=-1)
  return tokens



@torch.no_grad()
def sample(model, diffusion, config, num_samples=16):
    model.eval()
    x = torch.randn(
        (num_samples, config.context_length, config.hidden_size)
    ).to(config.device)
    for t in reversed(range(config.steps)):

        t_batch = torch.full(
            (num_samples,),
            t,
            device=config.device,
            dtype=torch.long
        )

        pred_x0 = model(x, t_batch)

        x = diffusion.denoise_step(x, pred_x0, t)

    return x

def decode_to_text(embeddings, model, tokenizer):

    vocab = model.embedding.weight
    vocab = F.normalize(vocab, dim=-1)
    embeddings = F.normalize(embeddings, dim=-1)

    similarity = torch.matmul(embeddings, vocab.T)

    probs = torch.softmax(similarity / 0.7, dim=-1)

    tokens = torch.multinomial(
        probs.view(-1, probs.size(-1)), 1
    ).view(probs.shape[:-1])

    return [
        tokenizer.decode(t, skip_special_tokens=True)
        for t in tokens
    ]

In [ ]:
dataloader = get_dataloader()

model = DiffusionTransformer(Config).to(Config.device) 
diffusion = Diffusion(Config.steps)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=Config.lr,
    weight_decay=Config.weight_decay
)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

for epoch in range(10):

    model.train()
    running_loss = 0

    loop = tqdm(dataloader, desc=f"Epoch {epoch}")

    for i, batch in enumerate(loop):

        tokens = batch["input_ids"].to(Config.device) #map the tokens

        with torch.no_grad():
            raw_embeddings = model.embed(tokens) #embed the tokens 
            target = raw_embeddings #set the embeddings as target

            timestamp = torch.randint(
                0,
                Config.steps,
                (tokens.size(0),),
                device=Config.device
            ) #get random timestamp to add noise

            x_t, _ = diffusion.add_noise(target, timestamp) #creates the corupted matrix. 

        pred_x0 = model(x_t, timestamp) #the model see the corrupted matrix and timestamp and guesses the correct embedding
        #it is not guessing noise

        loss = F.mse_loss(pred_x0, target) #add the loss

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        running_loss += loss.item()

        loop.set_postfix(
            loss=loss.item(),
            avg_loss=running_loss / (i + 1)
        )

    avg_epoch_loss = running_loss / len(dataloader)
    print(f"\nEpoch {epoch} Average Loss: {avg_epoch_loss:.4f}")


    model.eval()
    #Test after epoch
    print(f"Samples after Epoch {epoch}")

    with torch.no_grad():
        
        generated_embeddings = sample(
            model,
            diffusion,
            Config,
            num_samples=5
        )

        texts = decode_to_text(
            generated_embeddings,
            model,
            tokenizer
        )

        for i, t in enumerate(texts):
            print(f"Sample {i}: {t}\n")

    print("\n")